# Practice Lab #02 — Image Binary Classification with PyTorch
**Dataset:** Concrete Surface Crack Detection (`Positive` vs. `Negative`)  
**Runtime target:** Google Colab GPU (NVIDIA L4 / T4) or local machine.

This notebook orchestrates the modular `src/` package to fulfill every Lab 2 requirement:
1. Dataset loading, Stratified Train/Val/Test split, Data Augmentation & Preprocessing (`Resize → [0, 1] → Flatten 1D`).
2. Custom Activation Functions + exact Mathematical Derivatives wired into PyTorch Autograd.
3. Weight Initialization Convergence Comparison (`Random Normal` baseline vs. `Xavier` vs. `He`).
4. Alternative Activation Comparison (`ReLU`, `LeakyReLU`, `Tanh`).
5. Test Evaluation (`Accuracy %`, `Precision`, `Recall`, `F1-Score`, `ROC-AUC`, `Confusion Matrix`) & Report Visualizations.

## 1. Environment Setup (Google Colab L4 / Local) & Dataset Preparation

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/huylasiunhan/xlav.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # 1. Clone (or update) the GitHub repo directly on Colab SSD
    REPO_DIR = Path("/content/xlav")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

    SOURCE_DIR = REPO_DIR / "lab2" / "source"
    DRIVE_ROOT = Path("/content/drive/MyDrive/XLAV_Lab2")
    ZIP_PATH = DRIVE_ROOT / "surface_crack_dataset.zip"
    DATA_DIR = Path("/content/data")
    OUTPUT_DIR = DRIVE_ROOT / "outputs"

    # 2. Unzip dataset once to fast local Colab SSD (/content/data)
    if not (DATA_DIR / "Positive").is_dir():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_DIR)
        print(f"Unzipped dataset to {DATA_DIR}")
    else:
        print(f"Dataset already extracted at {DATA_DIR}")
else:
    SOURCE_DIR = Path.cwd()
    DATA_DIR = SOURCE_DIR.parent / "data"
    OUTPUT_DIR = SOURCE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

# Auto-select optimal DataLoader workers (Colab L4 has 12 vCPUs -> optimal num_workers = 4)
cpu_count = os.cpu_count() or 2
OPTIMAL_WORKERS = min(4, max(2, cpu_count // 2)) if torch.cuda.is_available() else 0

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"IN_COLAB        : {IN_COLAB}")
print(f"GPU             : {gpu_name}")
print(f"CPU vCores      : {cpu_count} -> Selected num_workers = {OPTIMAL_WORKERS}")
print(f"SOURCE_DIR      : {SOURCE_DIR}")
print(f"DATA_DIR        : {DATA_DIR}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

## 2. Configuration, Stratified Split & DataLoader Verification

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from src.activation import verify_activation_derivatives
from src.config import ExperimentConfig, seed_everything
from src.dataset import collect_image_paths_and_label, create_dataloaders, split_dataset
from src.metrics import format_metrics_summary
from src.model import ImageMLP
from src.trainer import run_experiment
from src.transform import build_augmentation_pipeline, build_preprocess_pipeline
from src.visualize import (
    plot_activation_functions,
    plot_augmentation_samples,
    plot_confusion_matrix_and_roc,
    plot_initialization_comparison,
    plot_loss_curves,
    plot_model_architecture,
)

# Base configuration shared across all fair comparison experiments
base_cfg = ExperimentConfig(
    data_dir=str(DATA_DIR),
    img_size=64,                  # 64x64x3 = 12,288 flattened input features
    num_channels=3,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    batch_size=64,                # Good gradient update frequency for 4,000 images on L4
    num_workers=OPTIMAL_WORKERS,  # 4 persistent workers on Colab L4 (12 vCPUs)
    hidden_dims=(512, 256, 128),
    activation_name="relu",
    init_method="he",
    dropout=0.2,
    lr=1e-3,
    epochs=25,
    seed=42,
)

seed_everything(base_cfg.seed)
print(base_cfg)
print(f"Flattened input_dim = {base_cfg.input_dim:,}")

# Inspect dataset distribution & split counts
all_paths, all_labels = collect_image_paths_and_label(base_cfg.data_dir)
(x_tr, y_tr), (x_va, y_va), (x_te, y_te) = split_dataset(
    all_paths, all_labels,
    base_cfg.train_ratio, base_cfg.val_ratio, base_cfg.test_ratio, base_cfg.seed,
)
print(f"Total images : {len(all_paths)} (Negative={all_labels.count(0)}, Positive={all_labels.count(1)})")
print(f"Train split  : {len(x_tr)} (Pos={sum(y_tr)}, Neg={len(y_tr) - sum(y_tr)})")
print(f"Val split    : {len(x_va)} (Pos={sum(y_va)}, Neg={len(y_va) - sum(y_va)})")
print(f"Test split   : {len(x_te)} (Pos={sum(y_te)}, Neg={len(y_te) - sum(y_te)})")

# Create DataLoaders and verify one mini-batch
train_loader, val_loader, test_loader = create_dataloaders(base_cfg)
xb, yb = next(iter(train_loader))
print(f"Batch X: shape={tuple(xb.shape)}, dtype={xb.dtype}, range=[{xb.min():.3f}, {xb.max():.3f}]")
print(f"Batch y: shape={tuple(yb.shape)}, dtype={yb.dtype}")

## 3. Visualization #1 — Input Samples Before vs. After Data Augmentation

In [ ]:
aug_pipeline = build_augmentation_pipeline()
print("Augmentation pipeline:", aug_pipeline)
print("Preprocess pipeline  :", build_preprocess_pipeline(base_cfg.img_size))

fig_aug = plot_augmentation_samples(
    all_paths,
    all_labels,
    aug_pipeline,
    num_per_class=2,
    save_path=str(OUTPUT_DIR / "01_augmentation_samples.png"),
)
plt.show()

## 4. Custom Activation Functions & Mathematical Derivative Verification

In [ ]:
# Verify analytical derivatives against central finite-difference approximations
derivative_errors = verify_activation_derivatives()
print("Max absolute error (Analytical vs. Numerical Derivative):")
for act_name, err in derivative_errors.items():
    print(f"  - {act_name:10s}: {err:.3e}")

fig_act = plot_activation_functions(
    save_path=str(OUTPUT_DIR / "02_activation_functions.png"),
)
plt.show()

## 5. Visualization #2 — Model Architecture Flowchart

In [ ]:
sample_model = ImageMLP.from_config(base_cfg)
print(sample_model)
print(f"Total trainable parameters: {sample_model.count_parameters():,}")

fig_arch = plot_model_architecture(
    sample_model,
    save_path=str(OUTPUT_DIR / "03_model_architecture.png"),
)
plt.show()

## 6. Experiment 1 — Weight Initialization Convergence Comparison
Compare **Random Normal (Baseline)** vs. **Xavier Normal** vs. **He Normal** under identical dataset, architecture (`ReLU`), optimizer, and random seed.

In [ ]:
init_experiments = {
    "Random Normal (Baseline)": replace(base_cfg, activation_name="relu", init_method="normal"),
    "Xavier Normal":            replace(base_cfg, activation_name="relu", init_method="xavier"),
    "He Normal":                replace(base_cfg, activation_name="relu", init_method="he"),
}

init_results = {}
for exp_name, exp_cfg in init_experiments.items():
    init_results[exp_name] = run_experiment(
        exp_name, exp_cfg, train_loader, val_loader, test_loader, verbose=True
    )

fig_init = plot_initialization_comparison(
    init_results,
    save_path=str(OUTPUT_DIR / "04_init_convergence_comparison.png"),
)
plt.show()

## 7. Experiment 2 — Alternative Activation Functions Comparison
Evaluate **ReLU + He**, **LeakyReLU + He**, and **Tanh + Xavier** (paired with their theoretically matched weight initializers).

In [ ]:
activation_results = {
    "ReLU + He": init_results["He Normal"],  # Reuse identical run from Experiment 1
    "LeakyReLU + He": run_experiment(
        "LeakyReLU + He",
        replace(base_cfg, activation_name="leaky_relu", init_method="he"),
        train_loader, val_loader, test_loader, verbose=True,
    ),
    "Tanh + Xavier": run_experiment(
        "Tanh + Xavier",
        replace(base_cfg, activation_name="tanh", init_method="xavier"),
        train_loader, val_loader, test_loader, verbose=True,
    ),
}

# Print unified summary table across all experiments for the Lab 2 report
all_results = {
    "ReLU + Random Normal": init_results["Random Normal (Baseline)"],
    "ReLU + Xavier":        init_results["Xavier Normal"],
    "ReLU + He":            activation_results["ReLU + He"],
    "LeakyReLU + He":       activation_results["LeakyReLU + He"],
    "Tanh + Xavier":        activation_results["Tanh + Xavier"],
}

header = f"{'Experiment':<24} | {'Best Ep':>7} | {'Test Acc (%)':>12} | {'Precision':>9} | {'Recall':>8} | {'F1-Score':>8} | {'ROC-AUC':>8} | {'Time(s)':>7}"
print(header)
print("-" * len(header))
for name, res in all_results.items():
    m = res.test_metrics
    print(
        f"{name:<24} | {res.history.best_epoch:>7d} | {m['accuracy_pct']:>11.2f}% | "
        f"{m['precision']:>9.4f} | {m['recall']:>8.4f} | {m['f1_score']:>8.4f} | "
        f"{m['roc_auc']:>8.4f} | {res.elapsed_seconds:>7.1f}"
    )

## 8. Visualization #3 (Loss Curve) & Final Test Evaluation of the Best Model

In [ ]:
# Select the best-performing model by Test F1-Score (tie-break by Accuracy)
best_name, best_res = max(
    all_results.items(),
    key=lambda kv: (kv[1].test_metrics["f1_score"], kv[1].test_metrics["accuracy_pct"]),
)
print(f"Best Model: {best_name} (Best Epoch = {best_res.history.best_epoch})")
print("-" * 45)
print(format_metrics_summary(best_res.test_metrics))

# Plot Training & Validation Loss / Accuracy curves
fig_loss = plot_loss_curves(
    best_res.history,
    title=f"Training & Validation Curves — {best_name}",
    save_path=str(OUTPUT_DIR / "05_best_loss_curves.png"),
)
plt.show()

# Plot Confusion Matrix & ROC Curve on Test set
fig_eval = plot_confusion_matrix_and_roc(
    best_res.test_metrics,
    title=f"Test Set Evaluation — {best_name}",
    save_path=str(OUTPUT_DIR / "06_test_cm_and_roc.png"),
)
plt.show()

print(f"All report figures saved to: {OUTPUT_DIR}")

## 9. Export Structured Results (`results_summary.json`) for Report Generation

In [ ]:
import json

# Export full numerical summary to JSON so report generation has exact figures without manual copy-paste
summary_payload = {
    "runtime": {
        "in_colab": IN_COLAB,
        "gpu": gpu_name,
        "cpu_vcores": cpu_count,
        "num_workers": OPTIMAL_WORKERS,
        "device": base_cfg.device,
    },
    "dataset": {
        "total_images": len(all_paths),
        "negative_count": all_labels.count(0),
        "positive_count": all_labels.count(1),
        "train_size": len(x_tr),
        "val_size": len(x_va),
        "test_size": len(x_te),
        "img_size": base_cfg.img_size,
        "input_dim": base_cfg.input_dim,
    },
    "derivative_max_errors": derivative_errors,
    "best_experiment": best_name,
    "experiments": {},
}

for exp_name, res in all_results.items():
    m = res.test_metrics
    summary_payload["experiments"][exp_name] = {
        "activation": res.config.activation_name,
        "init_method": res.config.init_method,
        "hidden_dims": list(res.config.hidden_dims),
        "dropout": res.config.dropout,
        "lr": res.config.lr,
        "epochs": res.config.epochs,
        "batch_size": res.config.batch_size,
        "num_params": res.num_params,
        "elapsed_seconds": round(res.elapsed_seconds, 2),
        "best_epoch": res.history.best_epoch,
        "best_val_loss": round(res.history.best_val_loss, 6),
        "train_losses": [round(v, 6) for v in res.history.train_losses],
        "val_losses": [round(v, 6) for v in res.history.val_losses],
        "train_accs": [round(v, 4) for v in res.history.train_accs],
        "val_accs": [round(v, 4) for v in res.history.val_accs],
        "test_metrics": {
            "test_loss": round(float(m["test_loss"]), 6),
            "accuracy_pct": round(float(m["accuracy_pct"]), 4),
            "precision": round(float(m["precision"]), 6),
            "recall": round(float(m["recall"]), 6),
            "f1_score": round(float(m["f1_score"]), 6),
            "roc_auc": round(float(m["roc_auc"]), 6),
            "confusion_matrix": m["confusion_matrix"],
        },
    }

summary_path = OUTPUT_DIR / "results_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved full experiment summary JSON to: {summary_path}")